#### Comparison of daily data through code

In [1]:
import os
import pandas as pd
import re
from datetime import datetime

# 🔧 Replace with your actual folder path
folder_path = r"C:\Users\IlaBarshilia\ACME Barricades\FDOT Web Scraping - FDOT Web Scraping Data\Output Files\November_2025_Cutoff"
output_file = r"C:\Users\IlaBarshilia\ACME Barricades\FDOT Web Scraping - FDOT Web Scraping Data\Output Files\FDOT_Daily_Comparison_Results_Nov16_CutOff.xlsx"

# 📅 Regex to extract date in format '2025-Aug-17'
date_pattern = re.compile(r'(\d{4}-[A-Za-z]{3}-\d{2})')

# 📂 Step 1: Collect and sort files by date
files = []
for filename in os.listdir(folder_path):
    match = date_pattern.search(filename)
    if match:
        try:
            file_date = datetime.strptime(match.group(1), "%Y-%b-%d")
            files.append((file_date, filename))
        except ValueError:
            print(f"⚠️ Skipping file with invalid date format: {filename}")

files.sort()  # Sort by date

# 🔑 Key columns for identifying rows
key_columns = ["Contract", "EstimateNumber", "Project", "Unit", "Item Code", "Previous"]
all_summaries = []

# 📊 Step 2: Compare each file with the next one and export results
with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    for i in range(len(files) - 1):
        date1, file1 = files[i]
        date2, file2 = files[i + 1]
        df1 = pd.read_excel(os.path.join(folder_path, file1), engine='openpyxl')
        df1 = df1.drop(columns=['EstimateEnd Date', 'Item Description'])
        df1=df1.drop_duplicates()
        df2 = pd.read_excel(os.path.join(folder_path, file2), engine='openpyxl')
        df2 = df2.drop(columns=['EstimateEnd Date', 'Item Description'])
        df2=df2.drop_duplicates()


        # Convert all values to string for consistent comparison
        df1_str = df1.astype(str).drop_duplicates()
        df2_str = df2.astype(str).drop_duplicates()

        # ✅ New rows: full rows in df2 not in df1
        new_mask = ~df2_str.apply(tuple, axis=1).isin(df1_str.apply(tuple, axis=1))
        new_rows = df2_str[new_mask]

        # 🗑️ Removed rows: full rows in df1 not in df2
        removed_mask = ~df1_str.apply(tuple, axis=1).isin(df2_str.apply(tuple, axis=1))
        old_rows = df1_str[removed_mask]

        # 🔄 Changed rows: same keys, different content
        df1_keyed = df1_str.set_index(key_columns)
        df2_keyed = df2_str.set_index(key_columns)

        common_index = df1_keyed.index.intersection(df2_keyed.index)
        df1_common = df1_keyed.loc[common_index]
        df2_common = df2_keyed.loc[common_index]

        # Align columns before comparison
        df2_common_aligned = df2_common.reindex(columns=df1_common.columns)

        # Compare aligned DataFrames
        changed_mask = (df1_common != df2_common_aligned).any(axis=1)
        changed_rows = df2_common_aligned[changed_mask].reset_index()

        # Remove changed rows from new and old tabs
        changed_tuples = changed_rows.apply(tuple, axis=1)

        # Convert new and old rows to tuples for comparison
        new_tuples = new_rows.apply(tuple, axis=1)
        old_tuples = old_rows.apply(tuple, axis=1)

        # Filter out rows that are also in changed_rows
        new_rows_filtered = new_rows[~new_tuples.isin(changed_tuples)]
        old_rows_filtered = old_rows[~old_tuples.isin(changed_tuples)]

        
        # 📝 Write to Excel with sheet names based on newer file's date
        sheet_date = date2.strftime('%Y-%b-%d')
        new_rows_filtered.to_excel(writer, sheet_name=f"{sheet_date}_new"[:31], index=False)
        old_rows_filtered.to_excel(writer, sheet_name=f"{sheet_date}_removed"[:31], index=False)
        changed_rows.to_excel(writer, sheet_name=f"{sheet_date}_changed"[:31], index=False)
        all_summaries.append({
            "Compared Files": [f"{file1} → {file2}"],
            "Date": [sheet_date],
            "New Rows": [len(new_rows_filtered)],
            "Removed Rows": [len(old_rows_filtered)],
            "Changed Rows": [len(changed_rows)]
        })
        summary_df = pd.DataFrame(all_summaries)
        summary_df.to_excel(writer, sheet_name="Summary", index=False)

print(f"\n✅ All comparisons complete. Results saved to: {output_file}")


✅ All comparisons complete. Results saved to: C:\Users\IlaBarshilia\ACME Barricades\FDOT Web Scraping - FDOT Web Scraping Data\Output Files\FDOT_Daily_Comparison_Results_Nov16_CutOff.xlsx


In [3]:

import os
import re
from datetime import datetime
import pandas as pd

# ---------- USER SETTINGS ----------
FOLDER = r"C:\Users\IlaBarshilia\ACME Barricades\FDOT Web Scraping - FDOT Web Scraping Data\Output Files\November_2025_Cutoff"
KEY_COLUMNS = ["Contract", "EstimateNumber", "Project", "Unit", "Item Code", "Previous", "This Est.", "To-Date"]
EXCLUDE_COLUMNS = ["This Est.", "To-Date"]

# >>> NEW: Globally exclude this/these column(s) from ALL analysis (not even allowed in keys)
EXCLUDE_GLOBAL = ["Date_Downloaded"]  #e.g., ["LastUpdated"]

# If reading Excel, choose sheet (None = first sheet)
EXCEL_SHEET_NAME = None

# ---------- HELPERS ----------
DATE_SUFFIX_RE = re.compile(r'(?P<date>\d{4}-[A-Za-z]{3}-\d{2})(?=\.(csv|xlsx|xls)$)', re.IGNORECASE)

def parse_date_from_filename(name: str):
    """
    Extract 'YYYY-Mon-DD' at the end of filename (before extension) and return a Python datetime.date.
    Example: 'report_2025-Nov-24.xlsx' -> date(2025, 11, 24)
    """
    m = DATE_SUFFIX_RE.search(name)
    if not m:
        return None
    date_str = m.group('date')
    try:
        dt = datetime.strptime(date_str, "%Y-%b-%d").date()
        return dt
    except ValueError:
        return None


def load_file(path: str):
    """
    Load CSV or Excel into a pandas DataFrame and DROP EXCLUDE_GLOBAL columns immediately.
    Handles sheet_name=None by selecting the first sheet automatically.
    """
    ext = os.path.splitext(path)[1].lower()
    if ext == ".csv":
        df = pd.read_csv(path)

    elif ext == ".xlsx":
        tmp = pd.read_excel(path, sheet_name=EXCEL_SHEET_NAME, engine="openpyxl")
        if isinstance(tmp, dict):
            # Pick the first sheet deterministically
            first_sheet_name = next(iter(tmp.keys()))
            df = tmp[first_sheet_name]
        else:
            df = tmp

    elif ext == ".xls":
        tmp = pd.read_excel(path, sheet_name=EXCEL_SHEET_NAME, engine="xlrd")
        if isinstance(tmp, dict):
            first_sheet_name = next(iter(tmp.keys()))
            df = tmp[first_sheet_name]
        else:
            df = tmp

    else:
        raise ValueError(f"Unsupported file type: {ext}")

    # Drop globally excluded columns so they never participate in analysis
    df = df.drop(columns=EXCLUDE_GLOBAL, errors="ignore")
    return df


def validate_columns(df: pd.DataFrame, keys, exclude_cols):
    """
    Ensure keys exist in df; adjust exclude list to those present.
    NOTE: keys are already sanitized to remove EXCLUDE_GLOBAL before this is called.
    """
    missing = [c for c in keys if c not in df.columns]
    if missing:
        raise ValueError(f"Missing key columns in file: {missing}")
    # exclude col may or may not exist; we will ignore those not present
    return [c for c in exclude_cols if c in df.columns]

def compare_two_days(prev_df: pd.DataFrame, curr_df: pd.DataFrame, keys, exclude_cols):
    """
    Returns:
      - new_count: rows present in curr but not in prev (by KEY_COLUMNS)
      - changed_count: rows where the keys exist in both, but some non-key/non-excluded columns differ
      - details: dict with DataFrames {'new_rows_keys', 'changed_rows', 'compare_columns'}
    """
    # Ensure only relevant columns are compared; allow different column sets per day
    prev_cols = set(prev_df.columns)
    curr_cols = set(curr_df.columns)
    common_cols = list(prev_cols & curr_cols)

    # We compare on columns that are common, excluding keys and exclude_cols
    ex_cols = [c for c in exclude_cols if c in common_cols]
    compare_cols = [c for c in common_cols if c not in keys and c not in ex_cols]

    # Deduplicate on keys to avoid false positives
    prev_key_df = prev_df[keys].drop_duplicates()
    curr_key_df = curr_df[keys].drop_duplicates()

    # New rows (anti-join): keys in current not in previous
    merged_new = curr_key_df.merge(prev_key_df, on=keys, how="left", indicator=True)
    new_rows_keys = merged_new[merged_new["_merge"] == "left_only"][keys]
    new_count = len(new_rows_keys)

    # Changed rows: inner join on keys, compare compare_cols
    if compare_cols:
        merged_compare = curr_df[keys + compare_cols].merge(
            prev_df[keys + compare_cols],
            on=keys,
            how="inner",
            suffixes=("_curr", "_prev")
        )
        # Build a mask where any compare_col differs for the row
        diff_mask = pd.Series(False, index=merged_compare.index)
        for c in compare_cols:
            a = merged_compare[f"{c}_curr"]
            b = merged_compare[f"{c}_prev"]
            # Treat NaNs as equal by filling with a sentinel
            sentinel = object()
            a_filled = a.fillna(sentinel)
            b_filled = b.fillna(sentinel)
            diff_mask |= (a_filled != b_filled)

        changed_rows = merged_compare.loc[diff_mask, keys + [f"{c}_prev" for c in compare_cols] + [f"{c}_curr" for c in compare_cols]]
        changed_count = len(changed_rows)
    else:
        # No comparable columns left after exclusions
        changed_rows = pd.DataFrame(columns=keys)
        changed_count = 0

    details = {
        "new_rows_keys": new_rows_keys,
        "changed_rows": changed_rows,
        "compare_columns": compare_cols
    }
    return new_count, changed_count, details

# ---------- MAIN ----------
def run_daily_comparison(folder: str, keys, exclude_cols):
    # Compute effective keys by removing any globally excluded columns
    keys_effective = [k for k in keys if k not in EXCLUDE_GLOBAL]
    if not keys_effective:
        raise ValueError("After applying EXCLUDE_GLOBAL, no key columns remain. Please adjust KEY_COLUMNS.")

    # Collect dated files
    files = []
    for fname in os.listdir(folder):
        path = os.path.join(folder, fname)
        if not os.path.isfile(path):
            continue
        if not fname.lower().endswith((".csv", ".xlsx", ".xls")):
            continue
        d = parse_date_from_filename(fname)
        if d:
            files.append((d, fname, path))

    if not files:
        raise RuntimeError("No dated CSV/XLSX/XLS files found ending with 'YYYY-Mon-DD' before the extension.")

    # Sort by date ascending
    files.sort(key=lambda x: x[0])

    summary_rows = []
    comparisons = []

    prev_date, prev_name, prev_path = None, None, None
    prev_df = None

    for (curr_date, curr_name, curr_path) in files:
        curr_df = load_file(curr_path)

        if prev_df is not None:
            # Validate columns and normalize exclude list for current day
            ex_prev = validate_columns(prev_df, keys_effective, exclude_cols)
            ex_curr = validate_columns(curr_df, keys_effective, exclude_cols)
            # Use union of available exclude columns for fairness
            ex_cols_union = list(set(ex_prev) | set(ex_curr))

            new_count, changed_count, details = compare_two_days(prev_df, curr_df, keys_effective, ex_cols_union)

            summary_rows.append({
                "Previous File": prev_name,
                "Current File": curr_name,
                "Previous Date": prev_date.isoformat(),
                "Current Date": curr_date.isoformat(),
                "New Rows (by keys)": new_count,
                "Changed Rows (excluding specified columns)": changed_count,
                "Compared Columns": ", ".join(details["compare_columns"])
            })

            comparisons.append({
                "pair": (prev_name, curr_name),
                "new_rows_keys": details["new_rows_keys"],
                "changed_rows": details["changed_rows"]
            })

        # advance
        prev_date, prev_name, prev_path = curr_date, curr_name, curr_path
        prev_df = curr_df

    # Build summary DataFrame
    summary_df = pd.DataFrame(summary_rows)
    return summary_df, comparisons

# ---------- USAGE ----------
if __name__ == "__main__":
    summary_df, comparisons = run_daily_comparison(FOLDER, KEY_COLUMNS, EXCLUDE_COLUMNS)

    # Print summary
    pd.set_option("display.max_colwidth", 200)
    print("\n=== Daily Comparison Summary ===")
    if summary_df.empty:
        print("No consecutive comparisons (need at least two dated files).")
    else:
        print(summary_df.to_string(index=False))

    # OPTIONAL: Save summary to Excel/CSV
    out_summary_excel = os.path.join(FOLDER, "Daily_Compare_Summary.xlsx")
    summary_df.to_excel(out_summary_excel, index=False)
    print(f"\nSaved summary to: {out_summary_excel}")

    # OPTIONAL: Save detailed changes per pair
    # (Uncomment if you want CSVs for each pair)
    # for comp in comparisons:
    #     prev_name, curr_name = comp["pair"]
    #     new_keys = comp["new_rows_keys"]
    #     changed = comp["changed_rows"]
    #     base = f"{os.path.splitext(prev_name)[0]}__vs__{os.path.splitext(curr_name)[0]}"
    #     new_csv = os.path.join(FOLDER, f"{base}__new_keys.csv")
    #     changed_csv = os.path.join(FOLDER, f"{base}__changed_rows.csv")
    #     new_keys.to_csv(new_csv, index=False)
    #     changed.to_csv(changed_csv, index=False)
    #     print(f"Saved: {new_csv} and {changed_csv}")


KeyboardInterrupt: 

In [7]:

import os
import re
from datetime import datetime
import pandas as pd

# ---------- USER SETTINGS ----------
FOLDER = r"C:\Users\IlaBarshilia\ACME Barricades\FDOT Web Scraping - FDOT Web Scraping Data\Output Files\November_2025_Cutoff"
KEY_COLUMNS = ["Contract", "EstimateNumber", "EstimateEnd Date", "Project", "Unit", "Item Code", "Previous"]
EXCLUDE_COLUMNS = ["This Est.", "To-Date"]

# >>> NEW: Globally exclude this/these column(s) from ALL analysis (not even allowed in keys)
EXCLUDE_GLOBAL = ["Date_Downloaded"]  #e.g., ["LastUpdated"]

# If reading Excel: sheet to read (None = auto-pick the first sheet)
EXCEL_SHEET_NAME = None

# ---------- HELPERS ----------
DATE_SUFFIX_RE = re.compile(r'(?P<date>\d{4}-[A-Za-z]{3}-\d{2})(?=\.(csv|xlsx|xls)$)', re.IGNORECASE)

def parse_date_from_filename(name: str):
    m = DATE_SUFFIX_RE.search(name)
    if not m:
        return None
    date_str = m.group('date')
    try:
        dt = datetime.strptime(date_str, "%Y-%b-%d").date()
        return dt
    except ValueError:
        return None

def load_file(path: str):
    ext = os.path.splitext(path)[1].lower()
    if ext == ".csv":
        df = pd.read_csv(path)

    elif ext == ".xlsx":
        tmp = pd.read_excel(path, sheet_name=EXCEL_SHEET_NAME, engine="openpyxl")
        df = tmp[next(iter(tmp.keys()))] if isinstance(tmp, dict) else tmp

    elif ext == ".xls":
        tmp = pd.read_excel(path, sheet_name=EXCEL_SHEET_NAME, engine="xlrd")
        df = tmp[next(iter(tmp.keys()))] if isinstance(tmp, dict) else tmp

    else:
        raise ValueError(f"Unsupported file type: {ext}")

    # Drop globally excluded column so it never participates in analysis
    if EXCLUDE_GLOBAL:
        df = df.drop(columns=EXCLUDE_GLOBAL, errors="ignore")
    return df


def parse_datetime_column(
    s: pd.Series,
    possible_formats=("%Y-%m-%d %H:%M:%S", "%Y-%m-%d", "%m/%d/%Y", "%d-%m-%Y"),
    dayfirst=False,):
    """
    Try vectorized parsing with common formats first; if none fits sufficiently,
    fallback to dateutil (coerce invalids to NaT).
    """
    s = s.copy()
    # If already datetime, just normalize timezone
    if pd.api.types.is_datetime64_any_dtype(s):
        return s.dt.tz_localize(None)

    # Try each known format and pick the one with the most successes
    best = None
    best_non_na = -1
    for fmt in possible_formats:
        try:
            parsed = pd.to_datetime(s, format=fmt, errors="coerce")
        except Exception:
            continue
        non_na = parsed.notna().sum()
        if non_na > best_non_na:
            best_non_na = non_na
            best = parsed
            best_fmt = fmt

    # If a format covers a substantial portion, use it and leave others as NaT
    if best is not None and best_non_na >= max(1, int(0.6 * s.notna().sum())):
        return best.dt.tz_localize(None)

    # Fallback: element-wise with dateutil — coerce invalids to NaT
    # NOTE: This may still warn; you can silence with contextmanager, or accept it.
    parsed = pd.to_datetime(s, errors="coerce", dayfirst=dayfirst)


def _normalize_df_for_compare(
    df: pd.DataFrame,
    common_cols: list[str],
    *,
    float_decimals: int = 6,
    strip_strings: bool = True,
    casefold_strings: bool = True,
    lower_strings: None | bool = None,  # backward compatibility
    na_sentinel: str = "__NA__"
) -> pd.DataFrame:
    """
    Normalize columns for reliable row-wise equality comparison:
    - Treat NaN as equal using a sentinel.
    - Strings: optional strip + casefold (or lower if requested).
    - Datetime: try parsing and make tz-naive if parsable.
    - Floats: round.
    """
    # Backward-compat bridge: if lower_strings is explicitly set, override casefold_strings
    if lower_strings is not None:
        casefold_strings = False  # we’ll use lower() instead

    out = df[common_cols].copy()

    for col in common_cols:
        s = out[col]

        # Try datetime conversion first for object-like columns
        if s.dtype == "object" or pd.api.types.is_datetime64_any_dtype(s):
            # Prefer vectorized parsing; fallback to coercion
            s_dt = pd.to_datetime(s, errors="coerce")
            # Treat as datetime if a reasonable portion parsed
            if s_dt.notna().sum() >= max(1, int(0.5 * s.notna().sum())):
                out[col] = s_dt.dt.tz_localize(None)
                continue

        # Numeric normalization
        if pd.api.types.is_numeric_dtype(s):
            out[col] = s.round(float_decimals) if pd.api.types.is_float_dtype(s) else s
            continue

        # String normalization
        s_obj = s.astype(str)
        if strip_strings:
            s_obj = s_obj.str.strip()

        if lower_strings:
            s_obj = s_obj.str.lower()
        elif casefold_strings:
            s_obj = s_obj.str.casefold()

        out[col] = s_obj

    # Make NaN comparable
    out = out.fillna(na_sentinel)
    return out


def full_row_sets(
    prev_df: pd.DataFrame,
    curr_df: pd.DataFrame,
    *,
    float_decimals: int = 6,
    strip_strings: bool = True,
    casefold_strings: bool = True,
    lower_strings: None | bool = None,
    na_sentinel: str = "__NA__"
):
    """
    Compare full rows across ALL common columns (after normalization).
    Returns: common_rows, unique_prev, unique_curr, common_cols
    """
    prev_cols = set(prev_df.columns)
    curr_cols = set(curr_df.columns)
    common_cols = sorted(prev_cols & curr_cols)
    if not common_cols:
        raise ValueError("No common columns between prev_df and curr_df.")

    prev_cmp = _normalize_df_for_compare(
        prev_df, common_cols,
        float_decimals=float_decimals,
        strip_strings=strip_strings,
        casefold_strings=casefold_strings,
        lower_strings=lower_strings,
        na_sentinel=na_sentinel
    ).drop_duplicates()

    curr_cmp = _normalize_df_for_compare(
        curr_df, common_cols,
        float_decimals=float_decimals,
        strip_strings=strip_strings,
        casefold_strings=casefold_strings,
        lower_strings=lower_strings,
        na_sentinel=na_sentinel
    ).drop_duplicates()

    prev_idx = prev_cmp.set_index(common_cols)
    curr_idx = curr_cmp.set_index(common_cols)

    common_index = prev_idx.index.intersection(curr_idx.index)
    common_rows = prev_idx.loc[common_index].reset_index()

    unique_prev_idx = prev_idx.index.difference(curr_idx.index)
    unique_curr_idx = curr_idx.index.difference(prev_idx.index)

    unique_prev = prev_idx.loc[unique_prev_idx].reset_index()
    unique_curr = curr_idx.loc[unique_curr_idx].reset_index()

    return common_rows, unique_prev, unique_curr, common_cols

def key_based_changes(
    prev_df: pd.DataFrame,
    curr_df: pd.DataFrame,
    keys,
    compare_only_cols=None
):
    """
    Using KEY_COLUMNS (keys):
      - New_Keys: keys present in curr not in prev
      - Removed_Keys: keys present in prev not in curr
      - Changed_Rows: keys present in both where *specified compare columns* differ.
        Returns a DataFrame with keys + prev vs curr values for each compared column,
        and a 'Changed_Columns' list indicating which of the compared columns changed.

    Parameters
    ----------
    prev_df : pd.DataFrame
    curr_df : pd.DataFrame
    keys : list[str] | tuple[str]
        Key columns that uniquely identify rows across both DataFrames.
    compare_only_cols : list[str] | None
        Columns to compare for changes (excluding keys).
        If None, defaults to ["This Est.", "To-Date"].

    Returns
    -------
    New_Keys : pd.DataFrame
        Distinct key rows that appear in curr_df but not in prev_df.
    Removed_Keys : pd.DataFrame
        Distinct key rows that appear in prev_df but not in curr_df.
    Changed_Rows : pd.DataFrame
        Rows where any of the specified compare columns changed.
        Contains: keys + compare columns (with suffixes _curr/_prev) + Changed_Columns.
    compared_columns_used : list[str]
        The subset of compare_only_cols that were actually present in both DataFrames.
    """

    # Defaults: compare exactly these two columns
    if compare_only_cols is None:
        compare_only_cols = ["This Est.", "To-Date"]

    # Validate keys exist in both
    missing_keys_prev = [k for k in keys if k not in prev_df.columns]
    missing_keys_curr = [k for k in keys if k not in curr_df.columns]
    if missing_keys_prev or missing_keys_curr:
        raise ValueError(
            f"Key column(s) missing. "
            f"Missing in prev: {missing_keys_prev}; Missing in curr: {missing_keys_curr}"
        )

    # Determine which of the requested compare columns exist in both
    compared_columns_used = [
        c for c in compare_only_cols
        if (c in prev_df.columns) and (c in curr_df.columns)
    ]

    # New/Removed keys (work on distinct keys)
    prev_keys = prev_df[keys].drop_duplicates()
    curr_keys = curr_df[keys].drop_duplicates()

    merged_new = curr_keys.merge(prev_keys, on=keys, how="left", indicator=True)
    New_Keys = merged_new[merged_new["_merge"] == "left_only"][keys]

    merged_removed = prev_keys.merge(curr_keys, on=keys, how="left", indicator=True)
    Removed_Keys = merged_removed[merged_removed["_merge"] == "left_only"][keys]

    # Changed rows: inner join on keys, compare ONLY the specified columns
    if compared_columns_used:
        merged_compare = curr_df[keys + compared_columns_used].merge(
            prev_df[keys + compared_columns_used],
            on=keys,
            how="inner",
            suffixes=("_curr", "_prev")
        )

        # Normalize to strings and treat NaNs as equal ("nan" == "nan")
        diff_mask = pd.Series(False, index=merged_compare.index)
        for c in compared_columns_used:
            a = merged_compare[f"{c}_curr"].astype(str)
            b = merged_compare[f"{c}_prev"].astype(str)
            diff_mask |= (a != b)

        changed = merged_compare.loc[diff_mask].copy()

        # Add a compact list of which compared columns changed for each row
        def list_changed_cols(row):
            changed_cols = []
            for c in compared_columns_used:
                if str(row[f"{c}_curr"]) != str(row[f"{c}_prev"]):
                    changed_cols.append(c)
            return ", ".join(changed_cols)

        if not changed.empty:
            changed["Changed_Columns"] = changed.apply(list_changed_cols, axis=1)
        Changed_Rows = changed
    else:
        # None of the requested compare columns exist in both
        Changed_Rows = pd.DataFrame(columns=keys + ["Changed_Columns"])

    return New_Keys, Removed_Keys, Changed_Rows, compared_columns_used


# ---------- MAIN ----------
def run_daily_comparison(folder: str, keys):
    # Collect dated files
    files = []
    for fname in os.listdir(folder):
        path = os.path.join(folder, fname)
        if not os.path.isfile(path):
            continue
        if not fname.lower().endswith((".csv", ".xlsx", ".xls")):
            continue
        d = parse_date_from_filename(fname)
        if d:
            files.append((d, fname, path))

    if not files:
        raise RuntimeError("No dated CSV/XLSX/XLS files found ending with 'YYYY-Mon-DD' before the extension.")

    # Sort by date ascending
    files.sort(key=lambda x: x[0])

    summary_rows = []
    prev_date, prev_name, prev_path = None, None, None
    prev_df = None

    # Output folder for per-pair Excel reports
    pair_out_dir = os.path.join(folder, "Daily_Compare_Reports")
    os.makedirs(pair_out_dir, exist_ok=True)

    for (curr_date, curr_name, curr_path) in files:
        curr_df = load_file(curr_path)

        if prev_df is not None:
            # --- Full-row set comparison across common columns
            common_rows, unique_prev, unique_curr, common_cols = full_row_sets(prev_df, curr_df)

            # --- Key-based changes
            New_Keys, Removed_Keys, Changed_Rows, compare_cols = key_based_changes(prev_df, curr_df, keys)

            # Build summary row
            summary_rows.append({
                "Previous File": prev_name,
                "Current File": curr_name,
                "Previous Date": prev_date.isoformat(),
                "Current Date": curr_date.isoformat(),
                "Total Rows (Prev)": len(prev_df),
                "Total Rows (Curr)": len(curr_df),
                "Common Rows (by all common columns)": len(common_rows),
                "Unique Rows in Prev (by all common columns)": len(unique_prev),
                "Unique Rows in Curr (by all common columns)": len(unique_curr),
                "New Rows (by keys)": len(New_Keys),
                "Removed Rows (by keys)": len(Removed_Keys),
                "Changed Rows (by keys, any non-key column)": len(Changed_Rows),
                "Columns Compared (non-key common)": ", ".join(compare_cols),
                "Global Column Dropped": EXCLUDE_GLOBAL
            })

            # Write per-pair Excel report
            prev_base = os.path.splitext(prev_name)[0]
            curr_base = os.path.splitext(curr_name)[0]
            safe_pair = f"{prev_base}__vs__{curr_base}"
            # Excel sheet names have 31-char limit; we'll keep short names for sheets and write columns.
            out_xlsx = os.path.join(pair_out_dir, f"{safe_pair}__diff.xlsx")

            with pd.ExcelWriter(out_xlsx, engine="openpyxl") as xw:
                # Add a small summary sheet at the top
                pair_summary = pd.DataFrame([{
                    "Previous File": prev_name,
                    "Current File": curr_name,
                    "Previous Date": prev_date.isoformat(),
                    "Current Date": curr_date.isoformat(),
                    "Total Rows (Prev)": len(prev_df),
                    "Total Rows (Curr)": len(curr_df),
                    "Common Rows (by common cols)": len(common_rows),
                    "Unique Rows in Prev (common cols)": len(unique_prev),
                    "Unique Rows in Curr (common cols)": len(unique_curr),
                    "New Rows (keys)": len(New_Keys),
                    "Removed Rows (keys)": len(Removed_Keys),
                    "Changed Rows (keys)": len(Changed_Rows),
                    "Global Column Dropped": EXCLUDE_GLOBAL,
                    "Key Columns": ", ".join(keys),
                    "Compared Non-Key Columns": ", ".join(compare_cols)
                }])
                pair_summary.to_excel(xw, index=False, sheet_name="Summary")

                # Common/unique sets
                common_rows.to_excel(xw, index=False, sheet_name="Common_Rows")
                unique_prev.to_excel(xw, index=False, sheet_name="Unique_in_Previous")
                unique_curr.to_excel(xw, index=False, sheet_name="Unique_in_Current")

                # Key-based: new/removed/changed
                New_Keys.to_excel(xw, index=False, sheet_name="New_Keys")
                Removed_Keys.to_excel(xw, index=False, sheet_name="Removed_Keys")
                Changed_Rows.to_excel(xw, index=False, sheet_name="Changed_Rows")

        # advance
        prev_date, prev_name, prev_path = curr_date, curr_name, curr_path
        prev_df = curr_df

    # Build summary DataFrame for all consecutive pairs
    summary_df = pd.DataFrame(summary_rows)

    # Save a global summary CSV
    out_summary_excel = os.path.join(pair_out_dir, "Daily_Compare_Summary.xlsx")
    summary_df.to_excel(out_summary_excel, index=False)

    return summary_df, os.path.join(folder, "Daily_Compare_Summary.xlsx"), pair_out_dir

# ---------- USAGE ----------
if __name__ == "__main__":
    summary_df, summary_csv_path, reports_dir = run_daily_comparison(FOLDER, KEY_COLUMNS)

    pd.set_option("display.max_colwidth", 200)
    print("\n=== Daily Comparison Summary ===")
    if summary_df.empty:
        print("No consecutive comparisons (need at least two dated files).")
    else:
        print(summary_df.to_string(index=False))
        print(f"\nSaved summary CSV: {summary_csv_path}")
        print(f"Per-pair Excel reports: {reports_dir}")


C:\Users\IlaBarshilia\AppData\Local\Temp\ipykernel_17592\3900014371.py:118: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  s_dt = pd.to_datetime(s, errors="coerce")
C:\Users\IlaBarshilia\AppData\Local\Temp\ipykernel_17592\3900014371.py:118: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  s_dt = pd.to_datetime(s, errors="coerce")
C:\Users\IlaBarshilia\AppData\Local\Temp\ipykernel_17592\3900014371.py:118: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  s_dt = pd.to_datetime(s, errors="coerce")
C:\Users\IlaBarshilia\AppData\Local\Temp\ipykernel_17592\3900014371.py:118: UserWarning:


=== Daily Comparison Summary ===
                    Previous File                      Current File Previous Date Current Date  Total Rows (Prev)  Total Rows (Curr)  Common Rows (by all common columns)  Unique Rows in Prev (by all common columns)  Unique Rows in Curr (by all common columns)  New Rows (by keys)  Removed Rows (by keys)  Changed Rows (by keys, any non-key column) Columns Compared (non-key common) Global Column Dropped
FDOT_Output_Data_2025-Nov-17.xlsx FDOT_Output_Data_2025-Nov-18.xlsx    2025-11-17   2025-11-18               5274               9221                                 5274                                            0                                         3947                3929                       0                                           0                This Est., To-Date     [Date_Downloaded]
FDOT_Output_Data_2025-Nov-18.xlsx FDOT_Output_Data_2025-Nov-19.xlsx    2025-11-18   2025-11-19               9221              10326                          

#### Compare Estimate Date for Each Contract and latest Estimate Number

In [92]:
latest_df = pd.read_excel(r"C:\Users\IlaBarshilia\ACME Barricades\FDOT Web Scraping - FDOT Web Scraping Data\Output Files\FDOT_Output_Data_2025-Aug-26.xlsx")
grouped_df=latest_df.groupby("Contract")["EstimateNumber"].max().reset_index()

grouped_df = grouped_df.merge(latest_df[['Contract', 'EstimateNumber', 'EstimateEnd Date']], how='left', on=['Contract', 'EstimateNumber']).drop_duplicates()
grouped = grouped_df.groupby(['Contract', 'EstimateNumber'])['EstimateEnd Date']
grouped_df['Count_EndDates'] = grouped.transform('nunique')
grouped_df['EstimateEnd Date'] = grouped.transform(lambda x:list(set(x)))
grouped_df=grouped_df.reset_index(drop=True)
trial=grouped_df[grouped_df['Count_EndDates']>1]
trial

,Contract,EstimateNumber,EstimateEnd Date,Count_EndDates
96,T2844,31,2025-07-16,3
97,T2844,31,2025-07-18,3
98,T2844,31,2025-07-17,3
135,T2A02,7,2025-07-30,2
136,T2A02,7,2025-07-31,2


##### Comparison without cut off date filter

In [113]:
grouped_df=filtered_df.groupby("Contract")["EstimateNumber"].max().reset_index()

grouped_df = grouped_df.merge(filtered_df[['Contract', 'EstimateNumber', 'EstimateEnd Date']], how='left', on=['Contract', 'EstimateNumber']).drop_duplicates()
grouped = grouped_df.groupby(['Contract', 'EstimateNumber'])['EstimateEnd Date']
grouped_df['Count_EndDates'] = grouped.transform('nunique')
grouped_df['EstimateEnd Date'] = grouped.transform(lambda x:list(set(x)))
grouped_df=grouped_df.reset_index(drop=True)
grouped_df = grouped_df.groupby(['Contract', 'EstimateNumber']).agg(max_date_diff_days = ('EstimateEnd Date', lambda x: (x.max() - x.min()).days)).reset_index()
grouped_df = grouped_df[grouped_df['max_date_diff_days']>=30]
grouped_df

,Contract,EstimateNumber,max_date_diff_days
24,E3V77,15,54
34,E3W54,8,65
53,E4U88,23,51
79,E53F7,4,35
101,E7N96,16,198
119,E8U91,12,30
171,T2907,23,159
231,T3711,35,35
237,T3735,42,149
243,T3783,10,35


In [114]:
# grouped_df.to_excel('Check_EstimateEnd Dates.xlsx', index=False)

filtered_df=filtered_df.merge(grouped_df, how='inner', on=['Contract', 'EstimateNumber'])
filtered_df.to_excel('Check_Estimate End Dates more than 30 days for latest estimate number.xlsx', index=False)

In [ ]:

trial =grouped_df[grouped_df['EstimateEnd Date']>=30]
trial

,Contract,EstimateNumber,EstimateEnd Date
24,E3V77,15,54
34,E3W54,8,65
53,E4U88,23,51
79,E53F7,4,35
101,E7N96,16,198
119,E8U91,12,30
171,T2907,23,159
231,T3711,35,35
237,T3735,42,149
243,T3783,10,35


In [ ]:
trial['Contract'].nunique()
# filtered_df.to_excel('Check_output.xlsx', index=False)
trial.to_excel

111

In [11]:
import pandas as pd
import os
print(os.getcwd())
df = pd.read_excel(r"C:\Users\IlaBarshilia\OneDrive - OIA Global\VS_Code\jobtran_jobmst_query.xlsx")
df.head()

c:\Users\IlaBarshilia\OneDrive - OIA Global\VS_Code


,job,trans_date,qty_complete,qty_scrapped,oper_num,item
0,js00130797,2022-10-28,0.00,11.0,70,75202FG
1,js00130797,2022-10-28,0.00,6.0,70,75202FG
2,js00131025,2022-10-28,0.00,10.0,65,16-1029-3
3,js00131025,2022-10-28,0.00,1.0,65,16-1029-3
4,cj00011004,2022-10-28,19.84,0.0,10,PG-F-PASTE-B


In [12]:
df2 = df.copy()
print(df2['job'].head(20))
df2['job'] = df2['job'].str.strip().str.lower()
print(df2['job'].head(20))
df2.drop(columns='trans_date', inplace=True)

0     js00130797
1     js00130797
2     js00131025
3     js00131025
4     cj00011004
5     cj00011004
6     cj00010885
7     cj00010885
8     cj00022584
9     cj00022584
10    sam2524389
11    jt03124314
12    JS00130948
13    js00130989
14    js00130989
15    JB00280642
16    jb00280508
17    JB00280508
18    JB00280706
19    JU00067741
Name: job, dtype: object
0     js00130797
1     js00130797
2     js00131025
3     js00131025
4     cj00011004
5     cj00011004
6     cj00010885
7     cj00010885
8     cj00022584
9     cj00022584
10    sam2524389
11    jt03124314
12    js00130948
13    js00130989
14    js00130989
15    jb00280642
16    jb00280508
17    jb00280508
18    jb00280706
19    ju00067741
Name: job, dtype: object


In [3]:
trial = df[df['job'] == 1072051]
trial

,job,trans_date,qty_complete,qty_scrapped,oper_num,item
197150,1072051,2023-05-26,-588.0,0.0,40,320-0281
197152,1072051,2023-05-26,0.0,128.0,50,320-0281
197153,1072051,2023-05-26,0.0,12.0,30,320-0281
197154,1072051,2023-05-26,12.0,0.0,20,320-0281
197155,1072051,2023-05-26,0.0,0.0,20,320-0281


In [4]:
# Group by multiple columns and calculate min/max of oper_num and difference of qty
result = (
    df.groupby(['job', 'trans_date', 'item'])
      .agg(
          MinOper=('oper_num', 'min'),
          MaxOper=('oper_num', 'max'),
          MinQty=('qty_complete', 'min'),
          MaxQty=('qty_complete', 'max'),
          qty_Scrapped=('qty_scrapped', 'sum')
      )
      .assign(Qty_Scrapped_Calc=lambda x: x['MaxQty'] - x['MinQty'])
      .reset_index()
)
result

,job,trans_date,item,MinOper,MaxOper,MinQty,MaxQty,qty_Scrapped,Qty_Scrapped_Calc
0,1072051,2023-05-26,320-0281,20,50,-588.00,12.00,140.0,600.0
1,1072052,2023-05-26,320-0281,10,40,0.00,128.00,128.0,128.0
2,1072056,2022-11-14,300-1182,20,30,78.00,78.00,0.0,0.0
3,EJ00121921,2022-12-19,B4899,10,10,0.00,0.00,1.0,0.0
4,J000000841,2022-11-04,110-0116,10,10,619.11,619.11,0.0,0.0
...,...,...,...,...,...,...,...,...,...
342769,sam6615293,2025-10-27,P0220-027-TUBE,40,50,86.00,120.00,51.0,34.0
342770,sam6615294,2025-10-24,P0220-020-TUBE,20,20,146.00,146.00,4.0,0.0
342771,sam6615294,2025-10-27,P0220-020-TUBE,40,50,111.00,146.00,35.0,35.0
342772,sam6615295,2025-10-24,P0220-015-TUBE,20,20,384.00,384.00,16.0,0.0


In [13]:
# Function to get qty for min and max oper_num
def qty_diff(group):
    min_row = group.loc[group['oper_num'].idxmin(), 'qty_complete']
    max_row = group.loc[group['oper_num'].idxmax(), 'qty_complete']
    return pd.Series({
        'MinOper': group['oper_num'].min(),
        'MaxOper': group['oper_num'].max(),
        'QtyAtMinOper': min_row,
        'QtyAtMaxOper': max_row,
        'Qty_Scrapped_Calc': max_row - min_row
    })

result2 = df2.groupby(['job', 'item']).apply(qty_diff).reset_index()
result2

C:\Users\IlaBarshilia\AppData\Local\Temp\ipykernel_18036\737505894.py:13: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  result2 = df2.groupby(['job', 'item']).apply(qty_diff).reset_index()


,job,item,MinOper,MaxOper,QtyAtMinOper,QtyAtMaxOper,Qty_Scrapped_Calc
0,cj00003915,PG-F-2016,10.0,10.0,39.90,39.90,0.0
1,cj00003919,PG-F-2029,10.0,10.0,161.38,161.38,0.0
2,cj00003932,PG-F-2059,10.0,10.0,20.28,20.28,0.0
3,cj00004421,PG-F-2035,10.0,10.0,856.50,856.50,0.0
4,cj00010645,PG-F-2078,10.0,10.0,119.71,119.71,0.0
...,...,...,...,...,...,...,...
69123,sam6615292,FSN0335,10.0,30.0,43.00,45.00,2.0
69124,sam6615293,P0220-027-TUBE,10.0,50.0,150.00,86.00,-64.0
69125,sam6615294,P0220-020-TUBE,10.0,50.0,150.00,111.00,-39.0
69126,sam6615295,P0220-015-TUBE,10.0,50.0,400.00,367.00,-33.0


In [14]:
df3 = df2.groupby(['job', 'item'])['qty_scrapped'].sum().reset_index()
df3

,job,item,qty_scrapped
0,cj00003915,PG-F-2016,0.0
1,cj00003919,PG-F-2029,0.0
2,cj00003932,PG-F-2059,0.0
3,cj00004421,PG-F-2035,0.0
4,cj00010645,PG-F-2078,0.0
...,...,...,...
69123,sam6615292,FSN0335,0.0
69124,sam6615293,P0220-027-TUBE,72.0
69125,sam6615294,P0220-020-TUBE,39.0
69126,sam6615295,P0220-015-TUBE,33.0


In [17]:
result3 = result2.merge(df3, how = 'left', on = ['job', 'item'])
result3=result3.drop_duplicates()
result3

,job,item,MinOper,MaxOper,QtyAtMinOper,QtyAtMaxOper,Qty_Scrapped_Calc,qty_scrapped
0,cj00003915,PG-F-2016,10.0,10.0,39.90,39.90,0.0,0.0
1,cj00003919,PG-F-2029,10.0,10.0,161.38,161.38,0.0,0.0
2,cj00003932,PG-F-2059,10.0,10.0,20.28,20.28,0.0,0.0
3,cj00004421,PG-F-2035,10.0,10.0,856.50,856.50,0.0,0.0
4,cj00010645,PG-F-2078,10.0,10.0,119.71,119.71,0.0,0.0
...,...,...,...,...,...,...,...,...
69123,sam6615292,FSN0335,10.0,30.0,43.00,45.00,2.0,0.0
69124,sam6615293,P0220-027-TUBE,10.0,50.0,150.00,86.00,-64.0,72.0
69125,sam6615294,P0220-020-TUBE,10.0,50.0,150.00,111.00,-39.0,39.0
69126,sam6615295,P0220-015-TUBE,10.0,50.0,400.00,367.00,-33.0,33.0


In [18]:
result3.to_excel('Scrapped_Quantities_Data.xlsx', index=False)

In [29]:
import pandas as pd

# Sample DataFrame
data = {
    'job': ['A', 'A', 'A', 'B', 'B'],
    'trans_date': ['2025-10-02', '2025-10-02', '2025-09-15', '2025-10-02', '2025-10-02'],
    'oper_num': [10, 20, 30, 15, 25],
    'qty': [40, 45, 47, 50, 55]
}

df = pd.DataFrame(data)
print(df)

# Convert trans_date to datetime
df['trans_date'] = pd.to_datetime(df['trans_date'])

# Group by multiple columns and calculate min/max of oper_num and difference of qty
result = (
    df.groupby(['job', 'trans_date'])
      .agg(
          MinOper=('oper_num', 'min'),
          MaxOper=('oper_num', 'max'),
          MinQty=('qty', 'min'),
          MaxQty=('qty', 'max')
      )
      .assign(QtyDiff=lambda x: x['MaxQty'] - x['MinQty'])
      .reset_index()
)

print(result)

  job  trans_date  oper_num  qty
0   A  2025-10-02        10   40
1   A  2025-10-02        20   45
2   A  2025-09-15        30   47
3   B  2025-10-02        15   50
4   B  2025-10-02        25   55
  job trans_date  MinOper  MaxOper  MinQty  MaxQty  QtyDiff
0   A 2025-09-15       30       30      47      47        0
1   A 2025-10-02       10       20      40      45        5
2   B 2025-10-02       15       25      50      55        5
